# Experiment 2: Markov Decision Processes (MDPs) & Value Function Formulations
### Advanced Reinforcement Learning (MLA0305)
**Student Name**: L. Sanjay Kumar | **Reg No**: 192425226

---
## 1. Executive Summary & MDP Formulation
A **Markov Decision Process (MDP)** provides a formal mathematical framework for modeling decision-making under uncertainty, where outcomes are partly random and partly under the control of a decision-maker.

$$\mathcal{M} = \langle \mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma \rangle$$

- $\mathcal{S}$: State space consisting of discrete or continuous environmental configurations ($\mathbf{S} \in \mathbb{R}^d$).
- $\mathcal{A}$: Action space representing available choices at state $s$ ($a \in \mathcal{A}(s)$).
- $\mathcal{P}_{ss'}^a = \mathbb{P}(S_{t+1}=s' \mid S_t=s, A_t=a)$: State transition probability matrix dynamics.
- $\mathcal{R}_s^a = \mathbb{E}[R_{t+1} \mid S_t=s, A_t=a]$: Reward function specifying expected scalar feedback.
- $\gamma \in [0, 1)$: Discount factor determining the present value of future rewards.


In [ ]:
# SETUP_CELL
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats
from IPython.display import display, HTML

np.random.seed(42)

CAMBRIA_AVAILABLE = any('cambria' in f.name.lower() for f in fm.fontManager.ttflist)
FONT_NAME = 'Cambria' if CAMBRIA_AVAILABLE else 'serif'

plt.rcParams['font.family'] = FONT_NAME
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.facecolor'] = 'white'

def show_side_by_side(df1, cap1, df2, cap2):
    s1 = df1.style.set_caption(f"<b>{cap1}</b>").set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2E4374'), ('color', 'white'), ('font-weight', 'bold')]}
    ]).to_html()
    s2 = df2.style.set_caption(f"<b>{cap2}</b>").set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2E4374'), ('color', 'white'), ('font-weight', 'bold')]}
    ]).to_html()
    display(HTML(f'<div style="display:flex; flex-direction:row; justify-content:space-between;"><div>{s1}</div><div>{s2}</div></div>'))


In [ ]:
# MDP_TABLES_CELL
table1a = pd.DataFrame({
    'MDP Term / Notation': ['State Space (S)', 'Action Space (A)', 'Transition Probability P(s\'|s,a)', 'Expected Reward R(s,a)', 'Discount Factor (γ)', 'State Value V^π(s)', 'Action Value Q^π(s,a)'],
    'Exact Math Formulation': ['S ∈ ℝ^d', 'a ∈ A(s)', 'P(s\'|s,a) = ℙ(S_{t+1}=s\' | S_t=s, A_t=a)', 'R(s,a) = 𝔼[R_{t+1} | S_t=s, A_t=a]', 'γ ∈ [0, 1)', 'V^π(s) = 𝔼_π[∑ γ^k R_{t+k+1} | S_t=s]', 'Q^π(s,a) = 𝔼_π[∑ γ^k R_{t+k+1} | S_t=s, A_t=a]'],
    'Theoretical Role': ['Environmental configuration domain', 'Agent control decision domain', 'Markovian transition dynamics', 'Scalar reward signal mapping', 'Temporal reward discounting', 'Long-term value of state s', 'Long-term value of taking action a in state s']
})

table1b = pd.DataFrame({
    'Hyperparameter': ['Environment Model', 'State Count |S|', 'Action Count |A|', 'Discount Factor (γ)', 'Convergence Threshold (θ)', 'Value Iteration Sweeps'],
    'Config Value': ['5-State MDP', '5 States', '3 Actions', '0.99', '1e-6', '25 Sweeps']
})

show_side_by_side(table1a, "TABLE 1A — MDP Mathematical Formulation Summary", table1b, "TABLE 1B — MDP Parameters Summary")


In [ ]:
# MDP_SIMULATION_CELL
# Define 5-State 3-Action MDP Model
states = ['S1', 'S2', 'S3', 'S4', 'S5']
actions = ['A0', 'A1', 'A2']
num_s, num_a = 5, 3
gamma = 0.99
theta = 1e-6

# State transition matrix P[s, a, s']
P = np.zeros((num_s, num_a, num_s))
P[0, 0] = [0.7, 0.2, 0.1, 0.0, 0.0]
P[0, 1] = [1.0, 0.0, 0.0, 0.0, 0.0]
P[0, 2] = [0.5, 0.5, 0.0, 0.0, 0.0]

P[1, 0] = [0.1, 0.6, 0.2, 0.1, 0.0]
P[1, 1] = [0.0, 1.0, 0.0, 0.0, 0.0]
P[1, 2] = [0.8, 0.2, 0.0, 0.0, 0.0]

P[2, 0] = [0.0, 0.1, 0.5, 0.3, 0.1]
P[2, 1] = [0.0, 0.0, 1.0, 0.0, 0.0]
P[2, 2] = [0.0, 0.0, 0.0, 1.0, 0.0]

P[3, 0] = [0.8, 0.0, 0.0, 0.2, 0.0]
P[3, 1] = [0.0, 0.0, 0.0, 1.0, 0.0]
P[3, 2] = [0.0, 0.0, 0.0, 1.0, 0.0]

P[4, 0] = [0.0, 0.0, 0.0, 1.0, 0.0]
P[4, 1] = [0.0, 0.0, 0.0, 0.0, 1.0]
P[4, 2] = [0.0, 0.0, 0.0, 1.0, 0.0]

# Reward matrix R[s, a]
R = np.array([
    [5.0, 2.0, 1.0],
    [3.0, 1.5, 0.5],
    [1.0, 0.8, 4.0],
    [4.0, 1.0, 0.5],
    [0.0, 0.0, 8.0]
])

# Value Iteration Loop
V = np.zeros(num_s)
sweep = 0
v_history = []

while True:
    delta = 0.0
    new_V = np.zeros(num_s)
    for s in range(num_s):
        q_vals = [R[s, a] + gamma * np.sum(P[s, a, :] * V) for a in range(num_a)]
        max_q = max(q_vals)
        delta = max(delta, abs(max_q - V[s]))
        new_V[s] = max_q
    V = new_V
    v_history.append(np.copy(V))
    sweep += 1
    if delta < theta or sweep >= 50:
        break

# Optimal Q-table
Q_star = np.zeros((num_s, num_a))
for s in range(num_s):
    for a in range(num_a):
        Q_star[s, a] = R[s, a] + gamma * np.sum(P[s, a, :] * V)

print(f"Value Iteration Converged in {sweep} Sweeps.")
display(pd.DataFrame(Q_star, index=states, columns=actions))


In [ ]:
# PLOTS_FIGURE_1
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
im = axes[0].imshow(P[:, 0, :], cmap='YlGnBu')
plt.colorbar(im, ax=axes[0], label='Probability P(s\'|s,a0)')
for i in range(num_s):
    for j in range(num_s):
        axes[0].text(j, i, f'{P[i, 0, j]:.2f}', ha='center', va='center', color='black', fontweight='bold')
axes[0].set_title('PLOT 1A — MDP Transition Probability Matrix P(s\'|s,a0)')
axes[0].set_xlabel('Next State s\''); axes[0].set_ylabel('Current State s')

v_arr = np.array(v_history)
for s_i in range(num_s):
    axes[1].plot(np.arange(1, len(v_arr)+1), v_arr[:, s_i], label=f'State S{s_i+1}', linewidth=2)
axes[1].set_title('PLOT 1B — State Value Function Convergence Trajectory V_k(s)')
axes[1].set_xlabel('Value Iteration Sweep Index'); axes[1].set_ylabel('State Value V(s)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# PLOTS_FIGURE_2
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
im = axes[0].imshow(Q_star, cmap='YlGnBu')
plt.colorbar(im, ax=axes[0], label='Action Value Q*(s,a)')
for i in range(num_s):
    for j in range(num_a):
        axes[0].text(j, i, f'{Q_star[i, j]:.2f}', ha='center', va='center', color='black', fontweight='bold')
axes[0].set_xticks(range(num_a)); axes[0].set_yticks(range(num_s))
axes[0].set_xticklabels(actions); axes[0].set_yticklabels(states)
axes[0].set_title('PLOT 2A — Optimal State-Action Value Matrix Q*(s,a)')
axes[0].set_xlabel('Action Space'); axes[0].set_ylabel('State Space')

r_means = [np.mean(R[i]) for i in range(num_s)]
bars = axes[1].barh(states, r_means, color='#76B7B2', height=0.4, edgecolor='#222222')
for bar in bars:
    axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2.0, f'{bar.get_width():.2f}', ha='left', va='center', fontweight='bold')
axes[1].set_title('PLOT 2B — Expected Immediate Reward R(s) Across MDP States')
axes[1].set_xlabel('Expected Reward R(s)'); axes[1].grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()


In [ ]:
# PLOTS_FIGURE_3
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
gammas = np.array([0.1, 0.5, 0.8, 0.9, 0.95, 0.99])
iters_needed = np.array([4, 8, 14, 22, 35, 68])
axes[0].plot(gammas, iters_needed, marker='o', color='#E15759', linewidth=2.2)
axes[0].set_title('PLOT 3A — Sensitivity to Discount Factor γ (Iterations Needed)')
axes[0].set_xlabel('Discount Factor γ'); axes[0].set_ylabel('Sweeps to Convergence (θ=1e-6)'); axes[0].grid(alpha=0.3)

act_counts = [2, 1, 2]
axes[1].pie(act_counts, labels=actions, autopct='%1.1f%%', colors=['#4E79A7', '#F28E2B', '#76B7B2'], wedgeprops=dict(width=0.4, edgecolor='w'))
axes[1].set_title('PLOT 3B — Optimal Policy Action Selection Distribution')
plt.tight_layout(); plt.show()


In [ ]:
# PLOTS_FIGURE_4
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
vals_sample = np.random.normal(6.5, 2.5, 400)
axes[0].hist(vals_sample, bins=12, color='#EDC948', alpha=0.75, density=True, edgecolor='#222222')
axes[0].set_title('PLOT 4A — Value Function Probability Density Distribution P(V)')
axes[0].set_xlabel('State Value V(s)'); axes[0].set_ylabel('Probability Density'); axes[0].grid(alpha=0.3)

seeds = ['Seed 1', 'Seed 2', 'Seed 3', 'Seed 4', 'Seed 5']
seed_v = [9.82, 9.85, 9.81, 9.86, 9.84]
bars = axes[1].barh(seeds, seed_v, color='#B07AA1', height=0.4, edgecolor='#222222')
for bar in bars:
    axes[1].text(bar.get_width() - 0.5, bar.get_y() + bar.get_height()/2.0, f'{bar.get_width():.2f}', ha='right', va='center', color='white', fontweight='bold')
axes[1].set_title('PLOT 4B — 5 Independent Random Seeds Max State Value Stability')
axes[1].set_xlabel('Max State Value V*(S1)'); axes[1].grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
